# 🔀 MATA Graph Control Flow — EarlyExit, While Loops & Conditional Edges

New in **v1.9.5**: the MATA graph scheduler supports three first-class
control-flow primitives that let you build smarter, more efficient pipelines:

| Primitive | What it does |
|-----------|-------------|
| `EarlyExit` | Halt the pipeline early when a predicate is met — skip all downstream nodes |
| `While` | Re-run a body of nodes in a do-while loop until a condition turns False |
| `Graph.add(condition=...)` | Attach a guard function to any node — skip it when the guard returns False |

This notebook covers each primitive with self-contained examples that
run without any external models (synthetic data + lightweight mocks), plus
a realistic end-to-end triage pipeline.

**Prerequisites**: `pip install datamata`

In [ ]:
import sys
sys.path.insert(0, "../../src")

import mata
from pathlib import Path

print(f"MATA version: {mata.__version__}")

IMAGE_1 = "../../examples/images/000000039769.jpg"
IMAGE_2 = "../../examples/images/000000015338.jpg"

for label, p in [("image 1", IMAGE_1), ("image 2", IMAGE_2)]:
    status = "✅ FOUND" if Path(p).exists() else "❌ NOT FOUND"
    print(f"  {label:10s}: {status}")

In [ ]:
"""Lightweight mock infrastructure used throughout this notebook.

All examples here run without downloading any model weights.
A `MockDetector` returns synthetic detections so you can focus on
understanding control-flow mechanics.
"""
from __future__ import annotations
from typing import Any

import numpy as np

from mata.core.artifacts.base import Artifact
from mata.core.artifacts.detections import Detections, Instance
from mata.core.artifacts.image import Image as MataImage
from mata.core.graph.context import ExecutionContext
from mata.core.graph.node import Node


# ---------------------------------------------------------------------------
# Tiny synthetic image helper
# ---------------------------------------------------------------------------

def make_image(h: int = 64, w: int = 64) -> MataImage:
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    return MataImage(data=arr)


# ---------------------------------------------------------------------------
# Synthetic Detections artifact
# ---------------------------------------------------------------------------

def make_detections(scores: list[float], labels: list[str] | None = None) -> Detections:
    """Return a Detections artifact with the given per-instance scores."""
    labels = labels or [f"obj_{i}" for i in range(len(scores))]
    instances = []
    for i, (score, label) in enumerate(zip(scores, labels)):
        inst = Instance(
            bbox=[10 * i, 10, 10 * i + 40, 50],
            score=score,
            label=0,
            label_name=label,
        )
        instances.append(inst)
    return Detections(instances=instances)


# ---------------------------------------------------------------------------
# A no-op detector node that returns configurable synthetic detections
# ---------------------------------------------------------------------------

class MockDetect(Node):
    """Returns fixed synthetic detections — no model required."""

    inputs: dict[str, Any] = {"image": MataImage}
    outputs: dict[str, Any] = {"dets": Detections}

    def __init__(self, scores: list[float], labels: list[str] | None = None, name: str = "detect"):
        super().__init__(name=name)
        self._dets = make_detections(scores, labels)

    def run(self, ctx: ExecutionContext, **kw) -> dict:
        ctx.store(f"{self.name}.dets", self._dets)
        print(f"  [{self.name}] produced {len(self._dets.instances)} detections")
        return {"dets": self._dets}


# ---------------------------------------------------------------------------
# A simple "log" node that records when it ran
# ---------------------------------------------------------------------------

class LogNode(Node):
    """Prints a message when it runs — useful for visualising skip behaviour."""

    inputs: dict[str, Any] = {}
    outputs: dict[str, Any] = {}

    def __init__(self, msg: str, name: str = "log"):
        super().__init__(name=name)
        self.msg = msg
        self.ran = False

    def run(self, ctx: ExecutionContext, **kw) -> dict:
        self.ran = True
        print(f"  [{self.name}] {self.msg}")
        return {}


print("✅ Mock infrastructure ready")

---

## 1️⃣ `EarlyExit` — Stop the Pipeline When a Condition Is Met

`EarlyExit` is a node that evaluates a **predicate** against the current
`ExecutionContext`.  If the predicate returns `True`, the scheduler receives
an `EarlyExitException` and stops processing all remaining nodes — returning
whatever results have been accumulated so far.

This is **not** an error.  It is an expected, low-cost control-flow signal.
The scheduler catches it cleanly and returns a partial `MultiResult`.

**Typical use-cases:**
- Quality gates: skip expensive inference when image quality is too low
- Triage: stop after a fast classifier already determined the outcome
- Alert pipelines: stop after dispatching an alert

```python
from mata.core.graph import EarlyExit

gate = EarlyExit(
    predicate=lambda ctx: len(ctx.retrieve("dets").instances) == 0,
    reason="No detections found — skipping segmentation",
    name="quality_gate",
)
```

In [ ]:
### 1a. EarlyExit node — standalone behaviour

from mata.core.graph import EarlyExit, EarlyExitException

ctx = ExecutionContext(providers={})

# ── Case 1: predicate is False → pipeline continues (returns empty dict) ──
gate_pass = EarlyExit(
    predicate=lambda ctx: False,
    reason="never triggered",
    name="gate_pass",
)
result = gate_pass.run(ctx)
print(f"Predicate=False  →  returns: {result!r}  (pipeline continues)")

# ── Case 2: predicate is True → EarlyExitException raised ──
gate_stop = EarlyExit(
    predicate=lambda ctx: True,
    reason="nothing to process",
    name="gate_stop",
)

try:
    gate_stop.run(ctx)
except EarlyExitException as e:
    print(f"Predicate=True   →  EarlyExitException: '{e.reason}' (node: {e.node_name})")

In [ ]:
### 1b. EarlyExit in a graph — nodes after the gate are not executed

from mata.core.graph import Graph, EarlyExit
from mata.core.graph.scheduler import SyncScheduler

print("=" * 55)
print("Scenario A — gate condition = True  (early stop)")
print("=" * 55)

detect_a   = MockDetect(scores=[], name="detect")            # 0 detections
after_gate = LogNode("running expensive segmentation", name="segment")

graph_a = (
    Graph("triage_a")
    .then(detect_a)
    .then(EarlyExit(
        predicate=lambda ctx: len(ctx.retrieve("detect.dets").instances) == 0,
        reason="No detections — skipping segmentation",
        name="gate",
    ))
    .then(after_gate)   # ← should NOT run
)

result_a = mata.infer(
    image=make_image(),
    graph=graph_a,
    providers={"detector": None},
)
print(f"\n  'segment' ran? → {after_gate.ran}")   # expected: False

print()
print("=" * 55)
print("Scenario B — gate condition = False  (pipeline continues)")
print("=" * 55)

detect_b   = MockDetect(scores=[0.9, 0.8], name="detect")   # 2 detections
after_gate2 = LogNode("running expensive segmentation", name="segment")

graph_b = (
    Graph("triage_b")
    .then(detect_b)
    .then(EarlyExit(
        predicate=lambda ctx: len(ctx.retrieve("detect.dets").instances) == 0,
        reason="No detections — skipping segmentation",
        name="gate",
    ))
    .then(after_gate2)   # ← SHOULD run
)

result_b = mata.infer(
    image=make_image(),
    graph=graph_b,
    providers={"detector": None},
)
print(f"\n  'segment' ran? → {after_gate2.ran}")   # expected: True

In [ ]:
### 1c. Visualise EarlyExit cost savings

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

stages_all  = ["Detect", "EarlyExit\ngate", "Segment", "Depth", "Fuse"]
stages_exit = ["Detect", "EarlyExit\ngate"]

fig, axes = plt.subplots(1, 2, figsize=(12, 2.5))
colors_all  = ["#4CAF50", "#FF9800", "#2196F3", "#9C27B0", "#009688"]
colors_exit = ["#4CAF50", "#FF9800"]

for i, (ax, stages, colors, title) in enumerate(zip(
    axes,
    [stages_all,  stages_exit],
    [colors_all,  colors_exit],
    ["Full pipeline (detections found)", "Early-exit pipeline (0 detections)"],
)):
    for j, (stage, color) in enumerate(zip(stages, colors)):
        rect = mpatches.FancyBboxPatch((j * 1.8, 0.2), 1.5, 0.6,
            boxstyle="round,pad=0.05", linewidth=1.5,
            edgecolor="black", facecolor=color, alpha=0.85)
        ax.add_patch(rect)
        ax.text(j * 1.8 + 0.75, 0.5, stage, ha="center", va="center",
                fontsize=8.5, fontweight="bold", color="white")
        if j < len(stages) - 1:
            ax.annotate("", xy=(j * 1.8 + 1.6, 0.5), xytext=(j * 1.8 + 1.5, 0.5),
                        arrowprops=dict(arrowstyle="->", color="black"))

    if i == 1:
        # Draw greyed-out skipped stages
        skipped = stages_all[len(stages_exit):]
        skip_colors = colors_all[len(stages_exit):]
        for j, (stage, color) in enumerate(zip(skipped, skip_colors)):
            x = (len(stages_exit) + j) * 1.8
            rect = mpatches.FancyBboxPatch((x, 0.2), 1.5, 0.6,
                boxstyle="round,pad=0.05", linewidth=1.5,
                edgecolor="#aaa", facecolor="#ccc", alpha=0.4, linestyle="--")
            ax.add_patch(rect)
            ax.text(x + 0.75, 0.5, stage, ha="center", va="center",
                    fontsize=8.5, color="#888")
            if j < len(skipped) - 1:
                ax.annotate("", xy=(x + 1.6, 0.5), xytext=(x + 1.5, 0.5),
                            arrowprops=dict(arrowstyle="->", color="#aaa"))

    ax.set_xlim(-0.2, len(stages_all) * 1.8)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=10, pad=4)

plt.suptitle("EarlyExit: halting a 5-stage pipeline after 2 stages", fontsize=11, y=1.05)
plt.tight_layout()
plt.show()

---

## 2️⃣ `While` — Bounded Loops for Iterative Refinement

`While` runs a **list of body nodes** in a loop.  After each complete pass
through the body, the *continuation condition* is evaluated:
- If it returns `True` → run the body again
- If it returns `False` (or `KeyError`) → stop

Execution has **do-while semantics** — the body always runs at least once.
A `max_iterations` cap prevents runaway loops.

```python
from mata.core.graph import While

loop = While(
    body=[
        RefineMask(src="masks", threshold=0.8, out="masks"),
        ScoreCheck(src="masks", out="quality"),
    ],
    condition=lambda ctx: ctx.retrieve("quality").score < 0.9,
    max_iterations=5,
    name="refine_loop",
)
```

**Typical use-cases:**
- Iterative mask refinement until quality threshold is met
- Adaptive confidence boosting (re-run NMS with tighter IoU each pass)
- Feedback loops that depend on the output of the previous iteration

In [ ]:
### 2a. While node — standalone mechanics

from mata.core.graph import While
from mata.core.artifacts.base import Artifact

# ── A minimal node that increments a counter stored on itself ──
class Counter(Node):
    """Increments an in-memory counter each time it runs."""

    inputs:  dict = {}
    outputs: dict = {}

    def __init__(self, name: str = "counter"):
        super().__init__(name=name)
        self.count = 0

    def run(self, ctx: ExecutionContext, **kw) -> dict:
        self.count += 1
        ctx.store("iteration", self.count)
        print(f"    iteration {self.count}")
        return {}


counter = Counter()
ctx = ExecutionContext(providers={})

# Run until counter reaches 3 (condition = "keep going while count < 3")
loop = While(
    body=[counter],
    condition=lambda ctx: ctx.retrieve("iteration") < 3,
    max_iterations=10,
    name="count_loop",
)

print("Running loop: continue while iteration < 3")
loop.run(ctx)
print(f"Final counter value: {counter.count}  (expected: 3)")
assert counter.count == 3

In [ ]:
### 2b. max_iterations cap — loop stops even if condition stays True

counter2 = Counter(name="counter2")
ctx2 = ExecutionContext(providers={})

loop_capped = While(
    body=[counter2],
    condition=lambda ctx: True,   # ← would loop forever without the cap
    max_iterations=4,
    name="capped_loop",
)

print("Running loop with max_iterations=4 and condition=always-True")
loop_capped.run(ctx2)
print(f"Final counter: {counter2.count}  (expected: 4 — hard cap triggered)")

In [ ]:
### 2c. Iterative confidence refinement simulation

# Simulate a detector that improves its max-confidence each pass
# (models the idea of iterative inference, e.g. prompt refinement)

class RefinementNode(Node):
    """Simulates iterative refinement: boosts confidence by `boost` each pass."""

    inputs:  dict = {}
    outputs: dict = {}

    def __init__(self, boost: float = 0.15, name: str = "refine"):
        super().__init__(name=name)
        self.boost = boost
        self.iteration = 0

    def run(self, ctx: ExecutionContext, **kw) -> dict:
        prev = ctx.retrieve("confidence") if ctx._artifacts.get("confidence") else 0.4
        # Explicitly store as plain float value inside a tiny custom artifact
        new_conf = min(prev + self.boost, 1.0)
        ctx.store("confidence", new_conf)
        self.iteration += 1
        print(f"    pass {self.iteration:02d}: confidence = {new_conf:.2f}")
        return {}


refine_node = RefinementNode(boost=0.15)
ctx3 = ExecutionContext(providers={})
ctx3.store("confidence", 0.4)

TARGET = 0.85

loop_refine = While(
    body=[refine_node],
    condition=lambda ctx: ctx._artifacts.get("confidence", 0.0) < TARGET,
    max_iterations=10,
    name="refine_loop",
)

print(f"Starting confidence: 0.40 — iterate until ≥ {TARGET}")
loop_refine.run(ctx3)

final_conf = ctx3._artifacts["confidence"]
print(f"\nFinal confidence after {refine_node.iteration} pass(es): {final_conf:.2f}")
print(f"Target {TARGET} reached: {'✅' if final_conf >= TARGET else '❌'}")

In [ ]:
### 2d. While in a full graph (detect → refine loop → log result)

from mata.core.graph import Graph, While

boost_node = RefinementNode(boost=0.2, name="boost")

graph_loop = (
    Graph("refinement_pipeline")
    .then(MockDetect(scores=[0.5, 0.4], name="detect"))
    .then(While(
        body=[boost_node],
        condition=lambda ctx: ctx._artifacts.get("confidence", 0.0) < 0.9,
        max_iterations=5,
        name="boost_loop",
    ))
    .then(LogNode("pipeline complete", name="done"))
)

ctx4 = ExecutionContext(providers={})
ctx4.store("confidence", 0.4)

print("Graph: detect → boost_loop (until conf ≥ 0.9) → done")
result_loop = mata.infer(
    image=make_image(),
    graph=graph_loop,
    providers={"detector": None},
)

print(f"\nBoost node ran {boost_node.iteration} time(s)")

---

## 3️⃣ Conditional Edges — `Graph.add(condition=...)`

Any node added to a graph with a `condition=` kwarg becomes **conditionally
executable**:

```python
graph = (
    Graph("smart_pipeline")
    .then(Detect(using="detector", out="dets"))
    .add(
        Segment(using="segmenter", out="masks"),
        condition=lambda ctx: len(ctx.retrieve("detect.dets").instances) > 2,
    )
)
```

When the scheduler reaches a node with a conditional edge it evaluates
the guard function.  If it returns `False` the node is **skipped** and added
to the `skipped_nodes` set.

### Cascade Skip

If a skipped node's outputs are wired as inputs to another node, that
downstream node is **automatically cascade-skipped** — you don't need to
add guards to downstream nodes manually.

```
detect  →  [segment: condition=False, SKIPPED]
                ↓
           [fuse: depends on segment.masks → CASCADE-SKIPPED]
```

In [ ]:
### 3a. Basic conditional edge — skip a node when condition is False

from mata.core.graph import Graph

for n_dets, expected_seg_ran in [(0, False), (3, True)]:
    segment_log = LogNode("running segmentation", name="segment")
    detect_node = MockDetect(scores=[0.9] * n_dets, name="detect")

    graph_cond = Graph(f"cond_edge_{n_dets}_dets")
    graph_cond.then(detect_node)
    graph_cond.add(
        segment_log,
        condition=lambda ctx: len(ctx.retrieve("detect.dets").instances) > 2,
    )

    mata.infer(image=make_image(), graph=graph_cond, providers={"detector": None})

    status = "✅" if segment_log.ran == expected_seg_ran else "❌"
    print(f"{status}  {n_dets} detections → segment ran: {segment_log.ran}  (expected: {expected_seg_ran})")

In [ ]:
### 3b. Cascade skip — nodes that depend on a skipped node are also skipped

class DependentNode:
    """Only runs if the upstream artifact exists."""
    def __init__(self, name="dependent"):
        self.name = name
        self.ran = False

    def __call__(self, ctx, **_):
        # Guard: require 'segment.masks' from the upstream segmentation node
        dets = ctx._artifacts.get("segment.masks")
        if dets is None:
            print(f"  [{self.name}] upstream 'segment' was skipped — nothing to do")
            return {}
        self.ran = True
        print(f"  [{self.name}] processing {dets} mask(s)")
        return {}


print("Scenario A: 0 detections → segment skipped → downstream guards itself")
segment_a = LogNode("segmenting", name="segment")
fuse_a    = DependentNode(name="fuse")
detect_a  = MockDetect(scores=[], name="detect")

g_cascade = Graph("cascade_skip")
g_cascade.then(detect_a)
g_cascade.add(
    segment_a,
    condition=lambda ctx: len(ctx.retrieve("detect.dets").instances) > 0,
)
g_cascade.then(fuse_a)

mata.infer(image=make_image(), graph=g_cascade, providers={"detector": None})
print(f"  segment ran: {segment_a.ran}, fuse ran: {fuse_a.ran}")  # False, False

print()
print("Scenario B: 4 detections → segment runs → downstream also runs")
segment_b = LogNode("segmenting", name="segment")
fuse_b    = DependentNode(name="fuse")
detect_b  = MockDetect(scores=[0.9, 0.8, 0.7, 0.6], name="detect")

g_b = Graph("cascade_run")
g_b.then(detect_b)
g_b.add(
    segment_b,
    condition=lambda ctx: len(ctx.retrieve("detect.dets").instances) > 0,
)
g_b.then(fuse_b)

mata.infer(image=make_image(), graph=g_b, providers={"detector": None})
print(f"  segment ran: {segment_b.ran}, fuse ran: {fuse_b.ran}")  # True, True

## 4. 🏗️ End-to-End Triage Pipeline

Combine all three primitives in a realistic **multi-stage object triage** pipeline:

```
Input frame
    │
    ▼
[detect]           ← regular detection node
    │
    ▼
[EarlyExit]        ← bail out when no objects found (skip expensive work)
    │  (dets found)
    ▼
[While: refine]    ← boost low-confidence predictions iteratively
    │  (max 5 passes)
    ▼
[segment]          ← conditional edge: only run if any det > 0.75 after refine
    │  (condition met)
    ▼
[log_result]       ← always runs if we reached this far
```

This pattern is common in production: skip frames with nothing interesting,
refine marginal detections, then only run the expensive segmentation model
when the refined confidence justifies the cost.

In [ ]:
from mata.core.graph import Graph, EarlyExit, While
from mata.core.artifacts import VisionResult

# ── helper nodes ─────────────────────────────────────────────────────────────

class RefineOnce:
    """Boost every detection score by +0.05 (capped at 1.0)."""
    def __init__(self, name="refine"):
        self.name = name
        self.calls = 0

    def __call__(self, ctx, **_):
        self.calls += 1
        prev = ctx.retrieve("detect.dets")
        boosted = [
            Instance(
                bbox=inst.bbox,
                score=min(1.0, inst.score + 0.05),
                label=inst.label,
            )
            for inst in prev.instances
        ]
        refined = VisionResult(instances=boosted)
        ctx.store("refined.dets", refined)
        scores = [f"{i.score:.2f}" for i in boosted]
        print(f"    refine pass {self.calls}: scores = {scores}")
        return {"refined.dets": refined}

    def loop_condition(self, ctx):
        """Keep refining while any detection is still below 0.75."""
        refined = ctx._artifacts.get("refined.dets") or ctx.retrieve("detect.dets")
        return any(i.score < 0.75 for i in refined.instances)


class SegmentNode:
    """Expensive segmentation — only called when condition is True."""
    def __init__(self, name="segment"):
        self.name = name
        self.ran = False

    def __call__(self, ctx, **_):
        self.ran = True
        refined = ctx._artifacts.get("refined.dets") or ctx.retrieve("detect.dets")
        print(f"  [segment] running on {len(refined.instances)} instances")
        return {}


def has_high_confidence(ctx):
    """Condition: at least one instance above threshold after refinement."""
    refined = ctx._artifacts.get("refined.dets") or ctx._artifacts.get("detect.dets")
    if refined is None:
        return False
    return any(i.score >= 0.75 for i in refined.instances)


# ── run two scenarios ─────────────────────────────────────────────────────────

for scenario, scores in [
    ("A — no detections (early exit)", []),
    ("B — low-conf dets that need refining", [0.60, 0.62]),
    ("C — already high-conf (no loop needed)", [0.90]),
]:
    print(f"\n{'='*55}")
    print(f"Scenario {scenario}")
    print('='*55)

    refine   = RefineOnce(name="refine")
    segment  = SegmentNode(name="segment")
    detect   = MockDetect(scores=scores, name="detect")

    graph = Graph("triage")
    graph.then(detect)
    graph.then(EarlyExit(
        condition=lambda ctx: len(ctx.retrieve("detect.dets").instances) == 0,
        message="No objects found — skipping expensive pipeline",
    ))
    graph.then(While(refine, condition=refine.loop_condition, max_iterations=5))
    graph.add(segment, condition=has_high_confidence)
    graph.then(LogNode("pipeline complete", name="done"))

    try:
        mata.infer(image=make_image(), graph=graph, providers={"detector": None})
    except SystemExit:
        pass  # EarlyExit raises SystemExit in standalone mode — graph handles it

    print(f"\n  refine calls : {refine.calls}")
    print(f"  segment ran  : {segment.ran}")

In [ ]:
### Visualise triage pipeline execution paths with matplotlib

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Triage Pipeline — Execution Paths", fontsize=14, fontweight="bold", y=1.02)

scenarios = [
    {
        "title": "Scenario A\n(no detections)",
        "nodes": ["detect", "EarlyExit", "While:refine", "segment", "done"],
        "active": [True, True, False, False, False],
        "exit_after": 1,
    },
    {
        "title": "Scenario B\n(low-conf → refine → segment)",
        "nodes": ["detect", "EarlyExit", "While:refine", "segment", "done"],
        "active": [True, True, True, True, True],
        "exit_after": None,
    },
    {
        "title": "Scenario C\n(high-conf, skip loop)",
        "nodes": ["detect", "EarlyExit", "While:refine", "segment", "done"],
        "active": [True, True, False, True, True],
        "exit_after": None,
    },
]

node_colors = {
    True:  "#4CAF50",   # green  = executed
    False: "#e0e0e0",   # grey   = skipped
}
EARLY_EXIT_COLOR = "#F44336"  # red for early-exit node

for ax, sc in zip(axes, scenarios):
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(sc["nodes"]) - 0.5)
    ax.axis("off")
    ax.set_title(sc["title"], fontsize=11)

    for i, (node, active) in enumerate(zip(sc["nodes"], sc["active"])):
        y = len(sc["nodes"]) - 1 - i
        color = (
            EARLY_EXIT_COLOR
            if node == "EarlyExit" and sc["exit_after"] == 1
            else node_colors[active]
        )
        rect = mpatches.FancyBboxPatch(
            (0.1, y - 0.35), 0.8, 0.65,
            boxstyle="round,pad=0.04",
            linewidth=1.2,
            edgecolor="#555",
            facecolor=color,
        )
        ax.add_patch(rect)
        ax.text(0.5, y - 0.03, node, ha="center", va="center",
                fontsize=9, fontweight="bold" if active else "normal",
                color="white" if active else "#888")

        if i < len(sc["nodes"]) - 1:
            next_active = sc["active"][i + 1]
            arrow_color = "#4CAF50" if next_active else "#e0e0e0"
            ax.annotate("", xy=(0.5, y - 0.35), xytext=(0.5, y - 0.65 + 0.35),
                        arrowprops=dict(arrowstyle="-|>", color=arrow_color, lw=1.5))

plt.tight_layout()
plt.show()

## 5. 🤖 Real-Model Demo (optional)

> **Requires model download** (~170 MB for `facebook/detr-resnet-50`).  
> Skip this section if you prefer to stay offline.

The same control-flow primitives work identically with real HuggingFace models —
just swap the mock nodes for real ones.

In [ ]:
# Uncomment to run (requires internet + ~170 MB model download)

# import mata
# from mata.core.graph import Graph, EarlyExit
# from mata.nodes import Detect
#
# IMAGE = "../../examples/images/000000039769.jpg"
#
# detector = mata.load("detect", "facebook/detr-resnet-50")
#
# graph = (
#     Graph("real_triage")
#     .then(Detect(name="detect", provider="detector"))
#     .then(EarlyExit(
#         condition=lambda ctx: len(ctx.retrieve("detect.dets").instances) == 0,
#         message="Empty frame — nothing to process",
#     ))
#     .then(LogNode("downstream processing", name="done"))
# )
#
# result = mata.infer(
#     image=IMAGE,
#     graph=graph,
#     providers={"detector": detector},
# )
# print(result)
print("Uncomment the cell above to run with a real DETR model.")

## ✅ Summary

This notebook demonstrated the three graph control-flow primitives added in v1.9.4:

| Feature | Import | Primary Use Case |
|---------|--------|-----------------|
| `EarlyExit` | `from mata.core.graph import EarlyExit` | Quality gate — bail out early on empty/low-quality frames |
| `While` | `from mata.core.graph import While` | Iterative refinement — repeat a node until a condition is met |
| `Graph.add(condition=...)` | built-in graph method | Cost gate — skip expensive nodes when confidence is too low |

### Key design principles

- **Composable** — all three primitives interoperate within a single `Graph`
- **Zero side effects** — skipped nodes leave no artifacts; downstream guards check what they need
- **Safe-by-default** — `While` always enforces `max_iterations` to prevent runaway loops
- **Backward compatible** — existing graphs without these primitives continue to work unchanged

### What's next?

- **v2.0.0** — `mata.export()` for quantized ONNX export, training pipeline
- **Parallel branches** — `Graph.branch()` for DAG-level concurrency
- **State across frames** — stateful nodes for video stream processing